### Create Snow-17 Forcing files from Met Station Data then filled with HRRR-AK Data: PPSA2

Notebook contents 
* Create Snow-17 forcing files, look at example structure, esp careful of precip units 

created by Cassie Lumbrazo\
last updated: Feb 2025\
run location: UAS linux\
python environment: **rasterio**

In [1]:
# import packages 
%matplotlib inline

# plotting packages 
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns 

sns.set_theme()
# plt.rcParams['figure.figsize'] = [12,6] #overriding size

# data packages 
import pandas as pd
import numpy as np
import xarray as xr
from datetime import datetime

import scipy

import rioxarray
import rasterio 
import cfgrib
import os

import geopandas as gpd

from zoneinfo import ZoneInfo
from pathlib import Path

In [2]:
pwd

'/home/cassie/python/repos/snow_model_forcing/sites/ppsa2'

## Open PPSA2 Met Station Data 

In [3]:
# download mesonet data as well to plot this with the Mesowest data...
filename = '/home/cassie/data/fishcreek/mesonet/rawdata/PowderPatch_everything_until_1June2025.txt'
df_met_raw = pd.read_csv(filename, sep = ",")

# make utc_valid a datetime and index the df_met by it 
df_met_raw['datetime'] = pd.to_datetime(df_met_raw['utc_valid'], utc=True)
df_met_raw = df_met_raw.set_index('datetime')

# create a dataframe, df_met, which contains only the datetime, PCIRZZZ, SDIRZZZ, TAIRZZZ, XRIRZZZ
df_met = df_met_raw[['PCIRZZZ', 'SDIRZZZ', 'TAIRZZZ', 'XRIRZZZ']].copy()

# cut to the same datetime as hrrr-ak
df_met = df_met.loc['2024-10-01T05:00':'2025-06-01T00:00']

# remove any PCIRZZZ values that are above 150
df_met['precip_accum_total'] = (df_met['PCIRZZZ'].where(df_met['PCIRZZZ'] <= 150)) * 25.4 # convert from inches to mm

# now, calculate the precipitation rate in mm/hr from the cumulative precip, and then convert to mm/s to compare with the model output
# df_met['precip_rate'] = df_met['precip_accum_total'].diff() / 1 # convert from mm/hr to mm/s (since the data is hourly, diff() gives us the change in precip per hour, so we divide by 1 to get mm/hr, then

# compute diff
diff = df_met["precip_accum_total"].diff()

# detect time gaps
time_diff = df_met.index.to_series().diff().dt.total_seconds()

# mask bad intervals
bad = (
    df_met["precip_accum_total"].isna() |
    df_met["precip_accum_total"].shift(1).isna() |
    (time_diff > 3600)
)

df_met["precip_accum_1hr"] = diff.mask(bad)

# get all the variables consistent first 
df_met['RH'] = df_met['XRIRZZZ']
df_met['HS'] = df_met['SDIRZZZ'] * 2.54 # convert from inches to mm, this is the snow depth in mm

# lightly clean HS so that no values are over 250cm 
df_met['HS'] = df_met['HS'].where(df_met['HS'] < 250, np.nan)

df_met['TA'] = (df_met['TAIRZZZ'] - 32) * 5/9 # convert from F to C
df_met.head()

,PCIRZZZ,SDIRZZZ,TAIRZZZ,XRIRZZZ,precip_accum_total,precip_accum_1hr,RH,HS,TA
datetime,,,,,,,,,
2024-10-01 05:00:00+00:00,278.62,17.08,37.2,92.6,NaN,NaN,92.6,43.3832,2.888889
2024-10-01 06:00:00+00:00,278.66,17.28,37.5,93.3,NaN,NaN,93.3,43.8912,3.055556
2024-10-01 07:00:00+00:00,278.66,17.08,38.2,93.8,NaN,NaN,93.8,43.3832,3.444444
2024-10-01 08:00:00+00:00,278.70,16.48,39.1,92.5,NaN,NaN,92.5,41.8592,3.944444
2024-10-01 09:00:00+00:00,278.70,17.28,39.7,91.5,NaN,NaN,91.5,43.8912,4.277778


## Load the HRRR Dataset, 
that we created in `2_clean_hrrrak_netcdf.ipynb`, a test for WY2024 for now

In [4]:
input_file = "/hdd/snow_hydrology/hrrrak/large_juneau_domain/netcdf/hrrrak_f567_WY2025_utm_site_ppsa2.nc"
ds = xr.open_dataset(input_file)
ds.rio.write_crs("EPSG:32608", inplace=True) # we know it is in UTM from the previous script work

ds['precip_accum_1hr'] = ds['precip_rate'] * 3600 # convert from mm/s to mm/hr
ds["precip_total_accum"] = ds.precip_accum_1hr.cumsum(dim="time") # total in mm accumulated precip over the time period
ds

<xarray.Dataset> Size: 2MB
Dimensions:             (time: 8760)
Coordinates:
  * time                (time) datetime64[ns] 70kB 2024-10-01T05:00:00 ... 20...
    valid_time          (time) datetime64[ns] 70kB ...
    step                (time) timedelta64[ns] 70kB ...
    y                   float64 8B ...
    x                   float64 8B ...
    spatial_ref         int64 8B 0
Data variables: (12/30)
    gust                (time) float64 70kB ...
    pressure            (time) float64 70kB ...
    orog                (time) float64 70kB ...
    temp_surface        (time) float64 70kB ...
    swe                 (time) float64 70kB ...
    snowdepth           (time) float64 70kB ...
    ...                  ...
    suswrf              (time) float64 70kB ...
    sulwrf              (time) float64 70kB ...
    wind                (time) float64 70kB ...
    wind_dir            (time) float64 70kB ...
    precip_accum_1hr    (time) float64 70kB 0.936 0.504 0.216 ... 0.0 0.0 0.0
    precip_total_accum  (time) float64 70kB 0.936 1.44 ... 2.976e+03 2.976e+03
Attributes: (12/13)
    long_name:       Wind speed (gust)
    units:           m s**-1
    GRIB_shortName:  gust
    GRIB_name:       Wind speed (gust)
    GRIB_cfName:     unknown
    GRIB_cfVarName:  gust
    ...              ...
    typeOfLevel:     surface
    site_name:       ppsa2
    original_lat:    58.262
    original_lon:    -134.517
    utm_x:           528340.7656649345
    utm_y:           6457981.62778889

# Create Snow-17 Input Data

In [5]:
# Powder Patch Site Info
# lat = 58.26200
# lon = -134.51700
# elevation = 669.95 # meters, from 2198ft 

### Mapping HRRR-AK Variables to Snow-17
| Variable    | HRRR units | Snow-17 units | Conversion   |
| ----------- | ---------- | ------------- | ------------ |
| temperature | K          | °C            | `K − 273.15` |
| precip_rate | kg m⁻² s⁻¹ | mm s⁻¹        | no change    |


## Check for missing dates and fill if missing 
Check there are no datetime gaps in the dataset, because Snow-17 is very strict and I've already had a problem with missing dates on 2023-10-29-00 <--- hour missing

In [6]:
def report_missing_hours(ds, time_var="time"):
    """
    Check for missing hourly timestamps in an xarray dataset.
    """
    time = pd.to_datetime(ds[time_var].values)

    # Expected hourly range
    full_range = pd.date_range(
        start=time.min(),
        end=time.max(),
        freq="1h"
    )

    missing = full_range.difference(time)

    if len(missing) == 0:
        print("No missing hourly timestamps.")
    else:
        print(f"Missing {len(missing)} hourly timestamps:")
        for t in missing:
            print(t)

    return missing

In [9]:
missing_times = report_missing_hours(ds)

No missing hourly timestamps.


## Create Forcing File

This version fills the missing data with interpolation, 

In [10]:
# Make ds time UTC-aware
# time = pd.to_datetime(ds.time.values).tz_localize("UTC")

# Or alternatively, strip timezone from df_met:
df_met.index = df_met.index.tz_convert(None)

In [11]:
# =========================
# SLICE DATASETS TO COMMON TIME RANGE
# =========================
start_time = '2024-10-01T05:00'
end_time = '2025-07-01T00:00'

# Slice met data
df_met = df_met.loc[start_time:end_time]

# Slice xarray dataset
ds = ds.sel(time=slice(start_time, end_time))

# =========================
# ALIGN TIME AXIS
# =========================
time_index = pd.to_datetime(ds.time.values)

# Ensure met index is datetime
df_met.index = pd.to_datetime(df_met.index)

# Resample met data to hourly
df_met_hourly = df_met.resample("1h").mean()

# Align to ds time
df_met_aligned = df_met_hourly.reindex(time_index)

# =========================
# CONVERT DS VARIABLES
# =========================
ds_temp_c = ds["temp"].values - 273.15          # K → °C
ds_prec_rate = ds["precip_rate"].values         # mm/s

# =========================
# CREATE FILLED DATAFRAME
# =========================
df_filled = pd.DataFrame(index=time_index)

# Temperature: use met, fill with ds
df_filled["tavg_degc"] = df_met_aligned["TA"].fillna(
    pd.Series(ds_temp_c, index=time_index)
)

# Precip:
# met gives accumulated mm per hour
# convert to mm/s for Snow-17
met_prec_rate = df_met_aligned["precip_accum_1hr"] / 3600.0

df_filled["prec_mm_s-1"] = met_prec_rate.fillna(
    pd.Series(ds_prec_rate, index=time_index)
)

# Round temperature
df_filled["tavg_degc"] = df_filled["tavg_degc"].round(2)

# Replace any remaining missing precip with zero
df_filled["prec_mm_s-1"] = df_filled["prec_mm_s-1"].fillna(0.0)

# =========================
# BUILD CONTINUOUS HOURLY INDEX
# =========================
df = df_filled.copy()

full_time = pd.date_range(
    start=df.index.min(),
    end=df.index.max(),
    freq="1h"
)

df = df.reindex(full_time)

# Fill temperature gaps
df["tavg_degc"] = df["tavg_degc"].interpolate(method="time")

# Fill precip gaps
df["prec_mm_s-1"] = df["prec_mm_s-1"].fillna(0.0)

# =========================
# PREPARE SNOW-17 FORMAT
# =========================
df = df.reset_index().rename(columns={"index": "time"})

df["year"] = df["time"].dt.year
df["mo"] = df["time"].dt.month
df["dy"] = df["time"].dt.day
df["hr"] = df["time"].dt.hour

df_out = df[[
    "year",
    "mo",
    "dy",
    "hr",
    "prec_mm_s-1",
    "tavg_degc"
]]

# =========================
# WRITE OUTPUT FILE
# =========================
output_file = "/home/cassie/python/models/run_snow17/sites/ppsa2/input/forcing/forcing.snow17bmi.met_ppsa2_WY2025.csv"

df_out.to_csv(
    output_file,
    index=False,
    float_format="%.6e"
)

print(f"Forcing file written to: {output_file}")

# =========================
# OPTIONAL DIAGNOSTIC PRINTS
# =========================
print("Fraction of timesteps using met data:")
print("TA:", df_met_aligned["TA"].notna().mean())
print("Precip:", df_met_aligned["precip_accum_1hr"].notna().mean())

Forcing file written to: /home/cassie/python/models/run_snow17/sites/ppsa2/input/forcing/forcing.snow17bmi.met_hrrak_ppsa2_WY2025.csv
Fraction of timesteps using met data:
TA: 0.8649969456322542
Precip: 0.8610262675626146


In [12]:
df_out

,year,mo,dy,hr,prec_mm_s-1,tavg_degc
0,2024,10,1,5,0.00026,2.89
1,2024,10,1,6,0.00014,3.06
2,2024,10,1,7,0.00006,3.44
3,2024,10,1,8,0.00000,3.94
4,2024,10,1,9,0.00009,4.28
...,...,...,...,...,...,...
6543,2025,6,30,20,0.00040,9.49
6544,2025,6,30,21,0.00010,9.87
6545,2025,6,30,22,0.00000,10.41
6546,2025,6,30,23,0.00010,9.70


In [13]:
# Check the final dataframe for any missing data 
df_out_rename = df_out.rename(columns={'mo':'month', 'dy':'day', 'hr':'hour'})
time = pd.to_datetime(df_out_rename[["year","month","day","hour"]]) 
dt = time.diff().dropna()

if not (dt == pd.Timedelta("1h")).all():
    raise ValueError("Forcing file still has time gaps.")
else:
    print("Forcing file has continuous hourly timestamps.")

Forcing file has continuous hourly timestamps.


## Create Params File

In [14]:
def write_snow17_params(
    output_file,
    hru_id,
    lat,
    lon,
    elev,
    area=1.0
):
    """
    Write a single-HRU Snow-17 parameter file (.txt).

    Parameters
    ----------
    hru_id : HRU identifier
    lat : latitude (deg)
    lon : longitude (deg) [not written to file, for metadata]
    elev : elevation (m)
    area : HRU area (km2), arbitrary for point runs
    """

    # Ensure .txt extension
    if not output_file.endswith(".txt"):
        output_file += ".txt"

    # Parameter values (typical default Snow-17 settings)
    params = [
        f"hru_id {hru_id}",
        f"hru_area {area}",
        f"latitude {lat}",
        f"elev {elev}",
        "scf 1.8",
        "mfmax 2.0",
        "mfmin 0.1",
        "uadj 0.05",
        "si 1500.0",
        "pxtemp 0.0",
        "nmf 0.15",
        "tipm 0.1",
        "mbase 0.0",
        "plwhc 0.03",
        "daygm 0.2",
        "adc1 0.05",
        "adc2 0.09",
        "adc3 0.16",
        "adc4 0.31",
        "adc5 0.54",
        "adc6 0.74",
        "adc7 0.84",
        "adc8 0.89",
        "adc9 0.93",
        "adc10 0.97",
        "adc11 1.00",
    ]

    with open(output_file, "w", encoding="ascii") as f:
        f.write("\n".join(params) + "\n")

    print(f"Snow-17 parameter file written to: {output_file}")

then use it, 

In [ ]:
write_snow17_params(
    output_file="/home/cassie/python/models/run_snow17/sites/ppsa2/input/params/snow17_params.met_ppsa2_WY2025.txt",
    hru_id="met_ppsa2_WY2025",
    lat=58.262,
    lon=-134.517,
    elev=670
)

Snow-17 parameter file written to: /home/cassie/python/models/run_snow17/sites/ppsa2/input/params/snow17_params.met_hrrrak_ppsa2_WY2025.txt


## Params File Info

### HRU descriptors 
| Parameter  | Meaning             | Units              |
| ---------- | ------------------- | ------------------ |
| `hru_id`   | HRU name/identifier | text               |
| `hru_area` | Area of HRU         | km² or model units |
| `latitude` | Latitude of HRU     | degrees            |
| `elev`     | Mean elevation      | m                  |


### Snowfall and melt parameters 
| Parameter | Meaning                                   | Typical range   |
| --------- | ----------------------------------------- | --------------- |
| `scf`     | Snow correction factor (gauge undercatch) | 0.8–2.5         |
| `mfmax`   | Max melt factor (non-rain)                | 0.5–5 mm/°C/day |
| `mfmin`   | Min melt factor                           | 0–1 mm/°C/day   |
| `uadj`    | Wind function during rain-on-snow         | 0–0.5           |
| `si`      | Snow water equivalent at 100% cover       | mm              |
| `pxtemp`  | Rain–snow threshold temperature           | °C              |


### Snowpack energy and mass parameters 
| Parameter | Meaning                             | Typical range |
| --------- | ----------------------------------- | ------------- |
| `nmf`     | Negative melt factor                | ~0.1–0.3      |
| `tipm`    | Antecedent temperature index weight | 0–1           |
| `mbase`   | Base melt temperature               | °C            |
| `plwhc`   | Liquid water holding capacity       | 0.01–0.1      |
| `daygm`   | Daily ground melt                   | 0–0.5 mm/day  |


### Areal depletion curve (ADC)
These describe fractional snow cover vs SWE
| Parameter      | Meaning                                      |
| -------------- | -------------------------------------------- |
| `adc1`–`adc11` | Snow cover fraction at increasing SWE levels |


# Create namelist file

In [16]:
def write_snow17_namelist(
    output_file,
    main_id,
    start_time,
    end_time,
    forcing_root="../input/forcing/forcing.snow17bmi.",
    output_root="../output/output.snow17bmi.",
    param_file="../input/params/snow17_params.",
    n_hrus=1,
    model_timestep=3600,
    output_hrus=0,
    warm_start_run=0,
    write_states=1
):
    """
    Write Snow-17 namelist file.

    Parameters
    ----------
    output_file : str
        Full path to output namelist file.
    main_id : str
        Station or basin identifier (e.g., "met_ppsa2_WY2025").
    start_time : str or pandas.Timestamp
        Start time (must convert to YYYYMMDDHH).
    end_time : str or pandas.Timestamp
        End time (must convert to YYYYMMDDHH).
    """

    import pandas as pd

    # Convert times to required format
    start_str = pd.to_datetime(start_time).strftime("%Y%m%d%H")
    end_str = pd.to_datetime(end_time).strftime("%Y%m%d%H")

    # Build parameter file path
    param_path = f"{param_file}{main_id}.txt"

    lines = [
        "&SNOW17_CONTROL",
        "! === run control file for snow17bmi v. 1.x ===",
        "",
        "! -- basin config and path information",
        f'main_id             = "{main_id}"',
        f"n_hrus              = {n_hrus}",
        f'forcing_root        = "{forcing_root}"',
        f'output_root         = "{output_root}"',
        f'snow17_param_file   = "{param_path}"',
        f"output_hrus         = {output_hrus}",
        "",
        "! -- run period information",
        f"start_datehr        = {start_str}",
        f"end_datehr          = {end_str}",
        f"model_timestep      = {model_timestep}",
        "",
        "! -- state start/write flags and files",
        f"warm_start_run      = {warm_start_run}",
        f"write_states        = {write_states}",
        "",
        "! -- filenames only needed if warm_start_run = 1",
        'snow_state_in_root  = "../state/snow17_states."',
        "",
        "! -- filenames only needed if write_states = 1",
        'snow_state_out_root = "../state/snow17_states."',
        "/"
    ]

    with open(output_file, "w", encoding="ascii") as f:
        f.write("\n".join(lines) + "\n")

    print(f"Snow-17 namelist written to: {output_file}")


Call that function to create the namelist file, 

In [ ]:
write_snow17_namelist(
    output_file="/home/cassie/python/models/run_snow17/sites/ppsa2/run/namelist.bmi.met_ppsa2_WY2025_function_test",
    main_id="met_ppsa2_WY2025",
    start_time=start_time,
    end_time=end_time
)

# Defined in the functions above, 
# start_time = '2024-10-01T05:00'
# end_time = '2025-07-01T00:00'

Snow-17 namelist written to: /home/cassie/python/models/run_snow17/sites/ppsa2/namelist.bmi.met_ppsa2_WY2025_function_test
